<a href="https://colab.research.google.com/github/syltaer-utp/s100-tareas/blob/main/S100_Equipo2_Parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S100 — Adelanto de modelado
## Baseline + árbol de regresión  
### Gasto en salud y esperanza de vida (América + Europa, 2000–2024)

**Equipo 2**

Este cuaderno cumple el adelanto del viernes 18:
- Baseline (predecir la media)
- Un modelo supervisado (árbol de regresión)
- Tabla comparativa de métricas (RMSE, MAE, R²)

## 1. Librerías

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Carga del dataset

Se carga `health_panel.csv` (panel país-año de World Bank, WHO y OECD).

In [3]:
DATA_PATH = "health_panel.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape original: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Años: {df['year'].min()} – {df['year'].max()}")
print(f"Países/entidades: {df['country_name'].nunique()}")

Shape original: 17,210 filas × 32 columnas
Años: 1960 – 2024
Países/entidades: 265


## 3. Construcción del subset

- Solo países de América y Europa (individuales).
- Solo años ≥ 2000.
- Solo filas con esperanza de vida y gasto en salud no nulos (sin eso no se puede plantear la hipótesis).
- Se elimina columnas vacías, redundantes y derivadas.
- Se crea la variable `continent`.

In [4]:
americas = [
    "Antigua and Barbuda", "Argentina", "Bahamas, The", "Barbados", "Belize",
    "Bolivia", "Brazil", "Canada", "Chile", "Colombia", "Costa Rica", "Cuba",
    "Dominica", "Dominican Republic", "Ecuador", "El Salvador", "Grenada",
    "Guatemala", "Guyana", "Haiti", "Honduras", "Jamaica", "Mexico",
    "Nicaragua", "Panama", "Paraguay", "Peru", "St. Kitts and Nevis",
    "St. Lucia", "St. Vincent and the Grenadines", "Suriname",
    "Trinidad and Tobago", "United States", "Uruguay", "Venezuela, RB",
]

europe = [
    "Albania", "Austria", "Belarus", "Belgium", "Bosnia and Herzegovina",
    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Hungary", "Iceland",
    "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Moldova", "Montenegro", "Netherlands", "North Macedonia", "Norway",
    "Poland", "Portugal", "Romania", "Russian Federation", "Serbia",
    "Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland",
    "Ukraine", "United Kingdom",
]

# Filtrar países y crear continente
df_ae = df[df["country_name"].isin(americas + europe)].copy()
df_ae["continent"] = np.where(df_ae["country_name"].isin(americas), "Americas", "Europe")

# Años >= 2000
df_ae = df_ae[df_ae["year"] >= 2000].copy()

# Quitar columnas vacías, redundantes y derivadas
cols_eliminar = [
    "region", "income_group", "iso2_code", "country_code",
    "le_per_gdp_point", "le_per_1k_spend", "le_spend_residual", "efficiency_score",
]
df_ae = df_ae.drop(columns=cols_eliminar, errors="ignore")

# Subset final: LE y gasto no nulos
df_subset = df_ae.dropna(
    subset=["life_expectancy_total", "health_spend_per_capita_usd"]
).copy()

print(f"df_subset: {df_subset.shape[0]:,} filas × {df_subset.shape[1]} columnas")
print(f"Años: {df_subset['year'].min()} – {df_subset['year'].max()}")
print(f"Países: {df_subset['country_name'].nunique()}")
print("\nPor continente:")
print(df_subset["continent"].value_counts())

df_subset: 1,786 filas × 25 columnas
Años: 2000 – 2024
Países: 75

Por continente:
continent
Europe      964
Americas    822
Name: count, dtype: int64


## 4. Definición de X e Y

- **y:** `life_expectancy_total` (continua → regresión)
- **X:** gasto en salud (hipótesis), PIB, año y continente (controles)

In [5]:
features = [
    "health_spend_per_capita_usd",
    "gdp_per_capita_usd",
    "year",
    "continent",
]

X = df_subset[features]
y = df_subset["life_expectancy_total"]

print(f"X: {X.shape} | y: {y.shape}")
print(f"Faltantes X:\n{X.isnull().sum()}")
print(f"Faltantes y: {y.isnull().sum()}")

X: (1786, 4) | y: (1786,)
Faltantes X:
health_spend_per_capita_usd    0
gdp_per_capita_usd             0
year                           0
continent                      0
dtype: int64
Faltantes y: 0


## 5. Train / test split

- 80 % entrenamiento, 20 % prueba.  
- `random_state=42` para reproducibilidad.  
- Unidad de observación: país-año (no se agrupa por año).

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")

Train: 1428 filas
Test:  358 filas


## 6. Baseline: predecir siempre la media

El baseline:

- Predice siempre el **promedio** de la esperanza de vida del conjunto de entrenamiento

Sirve como **piso de comparación**.  
Cualquier modelo útil debe tener RMSE y MAE **menores** que el baseline, y un R² claramente mayor que 0.


In [7]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred_b = baseline.predict(X_test)

rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_b))
mae_b  = mean_absolute_error(y_test, y_pred_b)
r2_b   = r2_score(y_test, y_pred_b)

print("Baseline (media)")
print(f"  RMSE: {rmse_b:.3f}")
print(f"  MAE:  {mae_b:.3f}")
print(f"  R²:   {r2_b:.3f}")

Baseline (media)
  RMSE: 4.464
  MAE:  3.716
  R²:   -0.011


## 7. Modelo supervisado: árbol de regresión

Se elige `DecisionTreeRegressor` porque:

1. La variable objetivo es **continua** → regresión  
2. Puede capturar relaciones **no lineales** (el efecto del gasto no tiene por qué ser una recta)  
4. Permite obtener **importancia de variables** (qué tanto aporta el gasto frente al PIB, año y continente)

**Preprocesamiento:**  
`continent` es categórica → **One-Hot Encoding** (sin imponer un orden artificial).

**Hiperparámetro:**  
`max_depth=5` limita la profundidad del árbol para reducir sobreajuste.  
Es un ajuste simple; en el reporte final se puede refinar con validación cruzada.

Todo va dentro de un `Pipeline` para que el preprocesamiento y el modelo se apliquen de forma consistente y sin fuga de información.

In [9]:
prep = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), ["continent"]),
    ("num", "passthrough", [
        "health_spend_per_capita_usd",
        "gdp_per_capita_usd",
        "year",
    ]),
])

modelo = Pipeline([
    ("prep", prep),
    ("tree", DecisionTreeRegressor(max_depth=5, random_state=42)),
])

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("Árbol de regresión (max_depth=5)")
print(f"  RMSE: {rmse:.3f}")
print(f"  MAE:  {mae:.3f}")
print(f"  R²:   {r2:.3f}")

Árbol de regresión (max_depth=5)
  RMSE: 2.458
  MAE:  1.864
  R²:   0.693
